# 总体方差的假设检验完整教程：卡方检验

## 📚 教学目标
1. 掌握卡方检验的假设设定和条件检查
2. 理解卡方检验统计量的计算公式
3. 用 scipy.stats.chi2 进行方差检验
4. 构建方差的置信区间并判断显著性
5. 理解方差检验的实际应用场景

**参考书**：《基础统计学(第14版)》（Triola）第 8-4 节
**教学策略**：先理解卡方分布的特征，再通过质量控制案例掌握完整的方差检验流程

---

## 1. 场景设定：为什么需要检验方差？

### 🎯 实际问题

某工厂声称其产品的方差不超过 0.0005。质检部门抽取 15 个样品进行检验。

**核心问题**：样本方差是否显著大于工厂声称的方差？

### 📖 书中的观点

> 方差和标准差的检验在质量控制中非常重要。
> 方差越小，产品质量越稳定一致。
> 我们使用卡方（χ²）分布来进行方差的假设检验。

### 📐 方差检验的核心公式

$$\chi^2 = \frac{(n-1)s^2}{\sigma_0^2}$$

其中：
- n = 样本量
- s² = 样本方差
- σ₀² = 假设的总体方差
- df = n - 1 = 自由度

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 卡方检验的条件与卡方分布

### 📐 两个必要条件

| 条件 | 说明 |
|------|------|
| 条件 1 | 样本为简单随机样本 |
| 条件 2 | **总体服从正态分布** |

### ⚠️ 重要提醒

卡方检验对正态性假设非常敏感！如果总体不是正态分布，卡方检验的结果可能不可靠。

### 📐 卡方分布的性质

1. **右偏分布**：始终为正值，右侧有长尾
2. **形状取决于 df**：df 越大，越接近对称
3. **非负值**：χ² ≥ 0
4. **极限性质**：当 df → ∞ 时，趋近正态分布

### 📐 检验类型

| 检验类型 | H₁ | p 值计算 |
|---------|-----|----------|
| 右侧检验 | σ² > σ₀² | 右侧面积 |
| 左侧检验 | σ² < σ₀² | 左侧面积 |
| 双侧检验 | σ² ≠ σ₀² | 2 × min(左侧, 右侧) |

In [ ]:
# ========== 卡方分布可视化 ==========

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图: 不同 df 的卡方分布
ax = axes[0]
x = np.linspace(0, 30, 1000)
dfs = [1, 3, 5, 10, 20]
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']

for df, color in zip(dfs, colors):
    y = stats.chi2.pdf(x, df)
    ax.plot(x, y, '-', color=color, linewidth=2, label=f'χ² (df={df})')

ax.set_xlabel('χ² Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Chi-Square Distribution for Different df', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_ylim(0, 0.5)

# 右图: 右侧检验示例
ax = axes[1]
df = 14
x = np.linspace(0, 35, 1000)
y = stats.chi2.pdf(x, df)
ax.plot(x, y, 'b-', linewidth=2, label=f'χ² Distribution (df={df})')
ax.fill_between(x, 0, y, alpha=0.1, color='steelblue')

# 标记临界值
alpha = 0.05
chi2_critical = stats.chi2.ppf(1 - alpha, df=df)
x_reject = np.linspace(chi2_critical, 35, 500)
ax.fill_between(x_reject, 0, stats.chi2.pdf(x_reject, df), alpha=0.4, color='#e74c3c',
                label=f'Rejection Region (α={alpha})')
ax.axvline(x=chi2_critical, color='red', linestyle='--', linewidth=2,
           label=f'Critical Value χ²={chi2_critical:.2f}')

# 标记检验统计量
chi2_stat = 22.5
ax.axvline(x=chi2_stat, color='#e67e22', linestyle='-', linewidth=2.5,
           label=f'Test Statistic χ²={chi2_stat:.2f}')

ax.set_xlabel('χ² Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Right-Tailed Chi-Square Test Example', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：不同自由度的卡方分布，df 越大越对称")
print("  右图：右侧检验示例，红色区域为拒绝域")
print("  卡方分布始终为正值，右侧有长尾")
print("  橙色线 = 检验统计量，红色虚线 = 临界值")

---

## 3. 右侧检验完整示例：产品质量控制

### 🎯 问题

某工厂声称其产品的方差 σ² ≤ 0.0005。质检部门抽取 15 个样品，测得样本方差 s² = 0.0008。

检验命题："产品的方差大于 0.0005"（α = 0.05）

### 📐 假设设定

$H_0: \sigma^2 = 0.0005$
$H_1: \sigma^2 > 0.0005$ （原命题，右侧检验）

### 📐 检验统计量

$$\chi^2 = \frac{(n-1)s^2}{\sigma_0^2} = \frac{(15-1) \times 0.0008}{0.0005} = 22.4$$

In [ ]:
# ========== 右侧检验: 产品质量控制 ==========

print("=" * 60)
print("📋 产品质量控制 - 右侧卡方检验")
print("=" * 60)

# 数据
n = 15
s_squared = 0.0008
sigma0_squared = 0.0005
alpha = 0.05
df = n - 1

print(f"\n📊 数据概况")
print(f"  样本量 n = {n}")
print(f"  样本方差 s² = {s_squared}")
print(f"  假设方差 σ₀² = {sigma0_squared}")
print(f"  自由度 df = {df}")

print(f"\n📊 步骤 1-3: 假设设定")
print(f"  H₀: σ² = {sigma0_squared}")
print(f"  H₁: σ² > {sigma0_squared} (原命题)")
print(f"  检验类型: 右侧检验")

print(f"\n📊 步骤 4: 显著性水平 α = {alpha}")

print(f"\n📊 步骤 5: 检查条件")
print(f"  条件1: 简单随机样本 ✓")
print(f"  条件2: 总体服从正态分布 (需假设或检验) ✓")

# 计算检验统计量
chi2_stat = (df * s_squared) / sigma0_squared
print(f"\n📊 步骤 6: 计算检验统计量")
print(f"  χ² = (n-1)s² / σ₀²")
print(f"  χ² = ({n}-1) × {s_squared} / {sigma0_squared}")
print(f"  χ² = {df} × {s_squared} / {sigma0_squared}")
print(f"  χ² = {chi2_stat:.4f}")

# 计算 p 值 (右侧检验)
p_value = 1 - stats.chi2.cdf(chi2_stat, df=df)
print(f"\n📊 步骤 6 (续): 计算 p 值")
print(f"  右侧检验: p 值 = P(χ² > {chi2_stat:.4f})")
print(f"  p 值 = {p_value:.4f}")

# 临界值
chi2_critical = stats.chi2.ppf(1 - alpha, df=df)
print(f"\n📊 步骤 6 (续): 临界值法")
print(f"  临界值 χ²_α = {chi2_critical:.4f}")
print(f"  检验统计量 χ² = {chi2_stat:.4f}")

# 判断
print(f"\n📊 步骤 7: 做出判断")
print(f"  p 值法: p = {p_value:.4f} {'< α → 拒绝 H₀' if p_value < alpha else '≥ α → 不能拒绝 H₀'}")
print(f"  临界值法: χ² = {chi2_stat:.4f} {'> χ²_α → 拒绝 H₀' if chi2_stat > chi2_critical else '≤ χ²_α → 不能拒绝 H₀'}")

# 结论
print(f"\n📊 步骤 8: 结论")
if p_value < alpha:
    print(f"  有足够的证据支持'产品方差大于 {sigma0_squared}'")
else:
    print(f"  没有足够的证据支持'产品方差大于 {sigma0_squared}'")

print("\n" + "=" * 60)

---

## 4. 左侧检验示例

### 🎯 问题

某新工艺声称可以降低产品方差。现有工艺的方差为 0.01。采用新工艺后，抽取 20 个样品，测得样本方差 s² = 0.006。

检验命题："新工艺的方差小于 0.01"（α = 0.05）

### 📐 假设设定

$H_0: \sigma^2 = 0.01$
$H_1: \sigma^2 < 0.01$ （原命题，左侧检验）

In [ ]:
# ========== 左侧检验: 新工艺方差 ==========

print("=" * 60)
print("📋 新工艺方差 - 左侧卡方检验")
print("=" * 60)

# 数据
n = 20
s_squared = 0.006
sigma0_squared = 0.01
alpha = 0.05
df = n - 1

print(f"\n📊 数据概况")
print(f"  样本量 n = {n}")
print(f"  样本方差 s² = {s_squared}")
print(f"  假设方差 σ₀² = {sigma0_squared}")
print(f"  自由度 df = {df}")

print(f"\n📊 假设设定")
print(f"  H₀: σ² = {sigma0_squared}")
print(f"  H₁: σ² < {sigma0_squared} (原命题)")
print(f"  检验类型: 左侧检验")

# 计算检验统计量
chi2_stat = (df * s_squared) / sigma0_squared
print(f"\n📊 计算检验统计量")
print(f"  χ² = (n-1)s² / σ₀² = {df} × {s_squared} / {sigma0_squared}")
print(f"  χ² = {chi2_stat:.4f}")

# 计算 p 值 (左侧检验)
p_value = stats.chi2.cdf(chi2_stat, df=df)
print(f"\n📊 计算 p 值")
print(f"  左侧检验: p 值 = P(χ² < {chi2_stat:.4f})")
print(f"  p 值 = {p_value:.4f}")

# 临界值
chi2_critical = stats.chi2.ppf(alpha, df=df)
print(f"\n📊 临界值法")
print(f"  临界值 χ²_{{1-α}} = {chi2_critical:.4f}")
print(f"  检验统计量 χ² = {chi2_stat:.4f}")

# 判断
print(f"\n📊 判断")
print(f"  p 值法: p = {p_value:.4f} {'< α → 拒绝 H₀' if p_value < alpha else '≥ α → 不能拒绝 H₀'}")
print(f"  临界值法: χ² = {chi2_stat:.4f} {'< 临界值 → 拒绝 H₀' if chi2_stat < chi2_critical else '≥ 临界值 → 不能拒绝 H₀'}")

# 结论
print(f"\n📊 结论")
if p_value < alpha:
    print(f"  有足够的证据支持'新工艺方差小于 {sigma0_squared}'")
else:
    print(f"  没有足够的证据支持'新工艺方差小于 {sigma0_squared}'")

print("\n" + "=" * 60)

---

## 5. 双侧检验示例

### 🎯 问题

某生产线的方差应为 0.002。抽取 12 个样品，测得样本方差 s² = 0.0035。

检验命题："生产线的方差不等于 0.002"（α = 0.05）

### 📐 假设设定

$H_0: \sigma^2 = 0.002$
$H_1: \sigma^2 \neq 0.002$ （双侧检验）

### 📐 双侧检验的 p 值

$$p\text{值} = 2 \times \min(P(\chi^2 < \chi^2_{\text{stat}}), P(\chi^2 > \chi^2_{\text{stat}}))$$

In [ ]:
# ========== 双侧检验: 生产线方差 ==========

print("=" * 60)
print("📋 生产线方差 - 双侧卡方检验")
print("=" * 60)

# 数据
n = 12
s_squared = 0.0035
sigma0_squared = 0.002
alpha = 0.05
df = n - 1

print(f"\n📊 数据概况")
print(f"  样本量 n = {n}")
print(f"  样本方差 s² = {s_squared}")
print(f"  假设方差 σ₀² = {sigma0_squared}")
print(f"  自由度 df = {df}")

print(f"\n📊 假设设定")
print(f"  H₀: σ² = {sigma0_squared}")
print(f"  H₁: σ² ≠ {sigma0_squared} (双侧检验)")

# 计算检验统计量
chi2_stat = (df * s_squared) / sigma0_squared
print(f"\n📊 计算检验统计量")
print(f"  χ² = (n-1)s² / σ₀² = {df} × {s_squared} / {sigma0_squared}")
print(f"  χ² = {chi2_stat:.4f}")

# 计算 p 值 (双侧检验)
p_left = stats.chi2.cdf(chi2_stat, df=df)
p_right = 1 - stats.chi2.cdf(chi2_stat, df=df)
p_value = 2 * min(p_left, p_right)
print(f"\n📊 计算 p 值")
print(f"  左侧面积 P(χ² < {chi2_stat:.4f}) = {p_left:.4f}")
print(f"  右侧面积 P(χ² > {chi2_stat:.4f}) = {p_right:.4f}")
print(f"  双侧 p 值 = 2 × min({p_left:.4f}, {p_right:.4f}) = {p_value:.4f}")

# 临界值
chi2_left = stats.chi2.ppf(alpha/2, df=df)
chi2_right = stats.chi2.ppf(1 - alpha/2, df=df)
print(f"\n📊 临界值法")
print(f"  左侧临界值 χ²_{{1-α/2}} = {chi2_left:.4f}")
print(f"  右侧临界值 χ²_{{α/2}} = {chi2_right:.4f}")
print(f"  检验统计量 χ² = {chi2_stat:.4f}")

# 判断
print(f"\n📊 判断")
print(f"  p 值法: p = {p_value:.4f} {'< α → 拒绝 H₀' if p_value < alpha else '≥ α → 不能拒绝 H₀'}")
in_rejection = chi2_stat < chi2_left or chi2_stat > chi2_right
print(f"  临界值法: χ² = {chi2_stat:.4f} {'在拒绝域内 → 拒绝 H₀' if in_rejection else '不在拒绝域内 → 不能拒绝 H₀'}")

# 结论
print(f"\n📊 结论")
if p_value < alpha:
    print(f"  有足够的证据拒绝'方差等于 {sigma0_squared}'的看法")
else:
    print(f"  没有足够的证据拒绝'方差等于 {sigma0_squared}'的看法")

print("\n" + "=" * 60)

---

## 6. 方差的置信区间

### 📐 置信区间公式

方差的置信区间为：

$$\frac{(n-1)s^2}{\chi^2_{\text{right}}} < \sigma^2 < \frac{(n-1)s^2}{\chi^2_{\text{left}}}$$

其中：
- $\chi^2_{\text{right}} = \chi^2_{\alpha/2, df}$（右侧临界值）
- $\chi^2_{\text{left}} = \chi^2_{1-\alpha/2, df}$（左侧临界值）

### 💡 置信区间与假设检验的关系

如果原假设中的方差 σ₀² 不在置信区间内，则拒绝 H₀。

In [ ]:
# ========== 方差置信区间 ==========

print("=" * 60)
print("📋 方差的置信区间")
print("=" * 60)

# 示例: 产品质量数据
n = 15
s_squared = 0.0008
confidence_level = 0.95
alpha = 1 - confidence_level
df = n - 1

print(f"\n📊 数据:")
print(f"  样本量 n = {n}")
print(f"  样本方差 s² = {s_squared}")
print(f"  置信水平 = {confidence_level*100:.0f}%")

# 计算临界值
chi2_left = stats.chi2.ppf(alpha/2, df=df)
chi2_right = stats.chi2.ppf(1 - alpha/2, df=df)

print(f"\n📊 临界值:")
print(f"  χ²_{{1-α/2}} = {chi2_left:.4f}")
print(f"  χ²_{{α/2}} = {chi2_right:.4f}")

# 计算置信区间
ci_lower = (df * s_squared) / chi2_right
ci_upper = (df * s_squared) / chi2_left

print(f"\n📊 置信区间:")
print(f"  下限 = (n-1)s² / χ²_{{α/2}} = {df} × {s_squared} / {chi2_right:.4f}")
print(f"  下限 = {ci_lower:.6f}")
print(f"  上限 = (n-1)s² / χ²_{{1-α/2}} = {df} × {s_squared} / {chi2_left:.4f}")
print(f"  上限 = {ci_upper:.6f}")
print(f"\n  {confidence_level*100:.0f}% 置信区间: ({ci_lower:.6f}, {ci_upper:.6f})")

# 标准差的置信区间
print(f"\n📊 标准差的置信区间:")
print(f"  ({np.sqrt(ci_lower):.6f}, {np.sqrt(ci_upper):.6f})")

# 与假设检验的关系
sigma0_squared = 0.0005
print(f"\n📊 与假设检验的关系:")
print(f"  H₀ 假设值 σ₀² = {sigma0_squared}")
print(f"  置信区间 = ({ci_lower:.6f}, {ci_upper:.6f})")
in_ci = ci_lower <= sigma0_squared <= ci_upper
print(f"  σ₀² {'在' if in_ci else '不在'}置信区间内 → {'不能拒绝 H₀' if in_ci else '拒绝 H₀'}")

print("\n" + "=" * 60)

---

## 7. 用 scipy.stats 验证

### 🔬 scipy 中的卡方检验

scipy 没有直接的"方差检验"函数，但我们可以用 `stats.chi2` 分布来完成所有计算。

In [ ]:
# ========== scipy 完整验证 ==========

print("=" * 60)
print("📋 scipy 完整验证: 三个示例")
print("=" * 60)

# 示例1: 右侧检验
print("\n📊 示例1: 产品质量控制 (右侧检验)")
n1, s2_1, sigma2_0_1 = 15, 0.0008, 0.0005
df1 = n1 - 1
chi2_1 = (df1 * s2_1) / sigma2_0_1
p_1 = 1 - stats.chi2.cdf(chi2_1, df=df1)
chi2_crit_1 = stats.chi2.ppf(0.95, df=df1)

print(f"  n={n1}, s²={s2_1}, σ₀²={sigma2_0_1}")
print(f"  χ² = {chi2_1:.4f}")
print(f"  临界值 = {chi2_crit_1:.4f}")
print(f"  p 值 = {p_1:.4f}")
print(f"  结论: {'拒绝 H₀' if p_1 < 0.05 else '不能拒绝 H₀'}")

# 示例2: 左侧检验
print("\n📊 示例2: 新工艺方差 (左侧检验)")
n2, s2_2, sigma2_0_2 = 20, 0.006, 0.01
df2 = n2 - 1
chi2_2 = (df2 * s2_2) / sigma2_0_2
p_2 = stats.chi2.cdf(chi2_2, df=df2)
chi2_crit_2 = stats.chi2.ppf(0.05, df=df2)

print(f"  n={n2}, s²={s2_2}, σ₀²={sigma2_0_2}")
print(f"  χ² = {chi2_2:.4f}")
print(f"  临界值 = {chi2_crit_2:.4f}")
print(f"  p 值 = {p_2:.4f}")
print(f"  结论: {'拒绝 H₀' if p_2 < 0.05 else '不能拒绝 H₀'}")

# 示例3: 双侧检验
print("\n📊 示例3: 生产线方差 (双侧检验)")
n3, s2_3, sigma2_0_3 = 12, 0.0035, 0.002
df3 = n3 - 1
chi2_3 = (df3 * s2_3) / sigma2_0_3
p_left_3 = stats.chi2.cdf(chi2_3, df=df3)
p_right_3 = 1 - stats.chi2.cdf(chi2_3, df=df3)
p_3 = 2 * min(p_left_3, p_right_3)

print(f"  n={n3}, s²={s2_3}, σ₀²={sigma2_0_3}")
print(f"  χ² = {chi2_3:.4f}")
print(f"  p 值 = {p_3:.4f}")
print(f"  结论: {'拒绝 H₀' if p_3 < 0.05 else '不能拒绝 H₀'}")

print(f"\n  ✅ 所有计算基于 scipy.stats.chi2 分布函数完成")

print("\n" + "=" * 60)

---

## 8. 可视化：卡方检验结果

### 📊 图示说明

通过可视化直观展示卡方检验的判断过程。

In [ ]:
# ========== 可视化三种检验 ==========

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 示例1: 右侧检验
ax = axes[0]
df1 = 14
chi2_stat1 = 22.4
chi2_crit1 = stats.chi2.ppf(0.95, df=df1)

x = np.linspace(0, 35, 1000)
y = stats.chi2.pdf(x, df1)
ax.plot(x, y, 'b-', linewidth=2, label=f'χ² (df={df1})')
ax.fill_between(x, 0, y, alpha=0.1, color='steelblue')
x_reject = np.linspace(chi2_crit1, 35, 500)
ax.fill_between(x_reject, 0, stats.chi2.pdf(x_reject, df1), alpha=0.4, color='#e74c3c',
                label=f'Rejection (α=0.05)')
ax.axvline(x=chi2_crit1, color='red', linestyle='--', linewidth=2,
           label=f'Critical χ²={chi2_crit1:.2f}')
ax.axvline(x=chi2_stat1, color='#e67e22', linestyle='-', linewidth=2.5,
           label=f'Statistic χ²={chi2_stat1:.2f}')
ax.set_xlabel('χ² Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Right-Tailed Test', fontsize=14, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.3)

# 示例2: 左侧检验
ax = axes[1]
df2 = 19
chi2_stat2 = 11.4
chi2_crit2 = stats.chi2.ppf(0.05, df=df2)

x = np.linspace(0, 35, 1000)
y = stats.chi2.pdf(x, df2)
ax.plot(x, y, 'b-', linewidth=2, label=f'χ² (df={df2})')
ax.fill_between(x, 0, y, alpha=0.1, color='steelblue')
x_reject = np.linspace(0, chi2_crit2, 500)
ax.fill_between(x_reject, 0, stats.chi2.pdf(x_reject, df2), alpha=0.4, color='#e74c3c',
                label=f'Rejection (α=0.05)')
ax.axvline(x=chi2_crit2, color='red', linestyle='--', linewidth=2,
           label=f'Critical χ²={chi2_crit2:.2f}')
ax.axvline(x=chi2_stat2, color='#e67e22', linestyle='-', linewidth=2.5,
           label=f'Statistic χ²={chi2_stat2:.2f}')
ax.set_xlabel('χ² Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Left-Tailed Test', fontsize=14, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.3)

# 示例3: 双侧检验
ax = axes[2]
df3 = 11
chi2_stat3 = 19.25
chi2_left3 = stats.chi2.ppf(0.025, df=df3)
chi2_right3 = stats.chi2.ppf(0.975, df=df3)

x = np.linspace(0, 30, 1000)
y = stats.chi2.pdf(x, df3)
ax.plot(x, y, 'b-', linewidth=2, label=f'χ² (df={df3})')
ax.fill_between(x, 0, y, alpha=0.1, color='steelblue')
x_reject_left = np.linspace(0, chi2_left3, 500)
ax.fill_between(x_reject_left, 0, stats.chi2.pdf(x_reject_left, df3), alpha=0.4, color='#e74c3c',
                label='Rejection Region')
x_reject_right = np.linspace(chi2_right3, 30, 500)
ax.fill_between(x_reject_right, 0, stats.chi2.pdf(x_reject_right, df3), alpha=0.4, color='#e74c3c')
ax.axvline(x=chi2_left3, color='red', linestyle='--', linewidth=2)
ax.axvline(x=chi2_right3, color='red', linestyle='--', linewidth=2,
           label=f'Critical χ²={chi2_left3:.2f}, {chi2_right3:.2f}')
ax.axvline(x=chi2_stat3, color='#e67e22', linestyle='-', linewidth=2.5,
           label=f'Statistic χ²={chi2_stat3:.2f}')
ax.set_xlabel('χ² Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Two-Tailed Test', fontsize=14, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图 (右侧检验): χ² 统计量 > 临界值 → 拒绝 H₀")
print("  中图 (左侧检验): χ² 统计量 > 临界值 → 不能拒绝 H₀")
print("  右图 (双侧检验): 显示两个拒绝域")
print("  红色区域 = 拒绝域 (α = 0.05)")
print("  绿色区域 = 接受域")
print("  橙色线 = 检验统计量")

---

## 📌 核心概念回顾

### 📌 卡方检验的条件
- **定义**: 简单随机样本 + 总体服从正态分布
- **公式**: $\chi^2 = \frac{(n-1)s^2}{\sigma_0^2}$, df = n - 1
- **含义**: 衡量样本方差与假设方差的比值

### 📌 三种检验方法
- **p 值法**: p < α → 拒绝 H₀
- **临界值法**: 统计量在拒绝域内 → 拒绝 H₀
- **置信区间法**: σ₀² 不在 CI 内 → 拒绝 H₀
- **特点**: 对于方差检验，三种方法等价

### 📌 方差的置信区间
- **下限**: $(n-1)s^2 / \chi^2_{\alpha/2}$
- **上限**: $(n-1)s^2 / \chi^2_{1-\alpha/2}$
- **标准差 CI**: 对方差 CI 开方

### 📌 卡方分布的性质
- **右偏**: 始终为正值，右侧有长尾
- **形状**: 取决于 df，df 越大越对称
- **非负**: χ² ≥ 0

### 🔗 完整流程
```
设定假设 → 检查正态性 → 计算 χ² → 求 p 值 → 判断 → 结论
    ↓           ↓          ↓        ↓       ↓       ↓
  H₀/H₁     QQ图/检验   公式    查表/软件  p<α?   非技术语言
```

### 📝 检验方法选择

| 检验目标 | 检验方法 | 条件 |
|---------|----------|------|
| 均值 (σ 已知) | z 检验 | 正态或 n>30 |
| 均值 (σ 未知) | t 检验 | 正态或 n>30 |
| 方差/标准差 | χ² 检验 | 正态分布 |

---

## ❌ 常见误区

### ❌ 误区 1: 用 t 检验来检验方差
**✓ 正确做法**: t 检验用于检验均值，不是方差。检验方差必须使用卡方检验。两者的检验统计量和分布完全不同。

### ❌ 误区 2: 忽略正态性假设
**✓ 正确做法**: 卡方检验对正态性假设非常敏感。如果总体不是正态分布，卡方检验的结果可能不可靠。小样本时尤其要注意，应先检验正态性。

### ❌ 误区 3: 混淆方差检验和标准差检验
**✓ 正确理解**: 方差检验和标准差检验使用相同的检验统计量，因为 σ² 和 σ 是一一对应的。拒绝方差的 H₀ 等价于拒绝标准差的 H₀，但公式中使用的是方差。

### ❌ 误区 4: 小样本时方差检验的功效很高
**✓ 正确理解**: 方差检验在小样本时功效较低，可能无法检测到实际存在的差异。增大样本量可以提高功效。

### ❌ 误区 5: 在不满足参数检验条件时误用参数方法
**✓ 正确做法**: 如果数据不满足正态性假设，不应使用卡方检验。可以考虑使用非参数方法，如自助法（bootstrap）来估计方差的置信区间。